In [1]:
import os
import polars as pl
from src.utils.config import setup_clickhouse_client
from src.utils.helper import convert_row
from src.utils.spark import get_spark

spark = get_spark()

ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')
client = setup_clickhouse_client(ch_user, ch_password, ch_host)

In [ ]:
# Fetch flights data
flights_query = "SELECT * FROM aviation_flights order by (dep_scheduled_time, dep_actual_time, arr_scheduled_time, arr_actual_time)"
flights_result = client.query(flights_query).result_rows
flights_columns = client.query(flights_query).column_names
flights_result = [convert_row(row) for row in flights_result]

# Create spark DataFrame
flights_pdf = pl.DataFrame(flights_result, schema=flights_columns, orient='row')

print(f"Total flight records retrieved: {len(flights_pdf)}")
flights_pdf.head(5)

In [ ]:
weather_query = "SELECT * FROM historical_weather_data order by date_observed"

weather_result = client.query(weather_query).result_rows
weather_result = [convert_row(row) for row in weather_result]
weather_columns = client.query(weather_query).column_names

weather_pdf = pl.DataFrame(weather_result, schema=weather_columns, orient='row')

print(f"Total weather records retrieved: {len(weather_pdf)}")
weather_pdf.head(5)

Total weather records retrieved: 966


In [ ]:
print('Prepared flights and weather data for merging.')

# Filter out cancelled and unknown status flights
valid_statuses = ['cancelled', 'unknown']
flights_filtered = flights_pdf.filter(~pl.col('status').is_in(valid_statuses)) \
                    .unique(subset=['flight_type', 'status', 'iata_number', 'airline_name', 
                                    'dep_scheduled_time', 'arr_scheduled_time', 'dep_actual_time', 'arr_actual_time'],
                                    keep='first')

print(f"Flight records after filtering: {len(flights_filtered)}")

In [ ]:
# Create hour columns for both departure and arrival times
flights_filtered = flights_filtered.with_columns(
    pl.col('dep_scheduled_time').str.to_datetime().dt.truncate('1h').alias('dep_hour'),
    pl.col('arr_scheduled_time').str.to_datetime().dt.truncate('1h').alias('arr_hour')
)

# Split into departure and arrival flights for conditional merging 
departure_flights = flights_filtered.filter(pl.col('flight_type') == 'departure')
arrival_flights = flights_filtered.filter(pl.col('flight_type') == 'arrival')

print(f"\nDeparture flights: {len(departure_flights)}, Arrival flights: {len(arrival_flights)}")
departure_flights.head(3)
arrival_flights.head(3)

In [ ]:
# need to trace why insert_time, date_observed, and weather_hour, weather_main, weather_desc are null after conversion to polars
weather_pdf = weather_pdf.with_columns(
    pl.col('date_observed').str.to_datetime().dt.truncate('1h').alias('weather_hour') 
)

# Deduplicate weather data: keep only one record per hour
# find out how to concatenate unique weather conditions and descriptions for duplicate hour/date entries
weather_pdf = weather_pdf.group_by(['weather_hour', 'date_observed']).agg(
    [
        pl.exclude(["id", "insert_time", "weather_main", "weather_desc", "weather_hour", "date_observed"]).mean(),

        pl.col("weather_main")
        .drop_nulls()
        .unique()
        .str.join(", ")
        .alias("weather_main"),

        pl.col("weather_desc")
        .drop_nulls()
        .unique()
        .str.join(", ")
        .alias("weather_desc"),
    ]
) 
print(f"Weather records by hour: {len(weather_pdf)}")

weather_pdf.head(3)

In [ ]:
# Merge departure flights with weather
# Departure: join dep_hour with weather_hour (not exceeding scheduled time means weather_hour <= dep_hour)
if len(departure_flights) > 0:
    merged_departures = departure_flights.join(
        weather_pdf,
        how='left',
        left_on='dep_hour',
        right_on='weather_hour',
        maintain_order='left'
    )
    print(f"Merged departure flights: {len(merged_departures)}") # will need to check this tmr

# Merge arrival flights with weather
# Arrival: join arr_hour with weather_hour (same hour)
if len(arrival_flights) > 0:
    merged_arrivals = arrival_flights.join(
        weather_pdf,
        how='left',
        left_on='arr_hour',
        right_on='weather_hour',
        maintain_order='left'
    )
    print(f"Merged arrival flights: {len(merged_arrivals)}")

# Combine both datasets
if len(merged_departures) > 0 and len(merged_arrivals) > 0:
    combined_flights = pl.concat([merged_departures, merged_arrivals], how='vertical') \
            .sort(by=['dep_scheduled_time', 'dep_actual_time', 'arr_scheduled_time', 'arr_actual_time'], nulls_last=True)
    # print(combined_flights.columns)
    combined_flights = combined_flights.drop(['id', 'insert_time', 'iata_number', 'airline_name', 'codeshared_airline', 'codeshared_flight_number'])
else:
    combined_flights = pl.DataFrame()

print(f"\n Combined flights with weather data: {len(combined_flights)} rows")
combined_flights.head(3)